In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

# 1.3 RAG

In [2]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [3]:
llm("Hey, what's up?")

'Hey! Not much — just here and ready to help. What’s going on with you?'

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.4-mini in organization org-gSJCDct1Mm7TXK9ly8OLI3td on requests per day (RPD): Limit 50, Used 50, Requested 1. Please try again in 28m48s. Visit https://platform.openai.com/account/rate-limits to learn more. You can increase your rate limit by adding a payment method to your account at https://platform.openai.com/account/billing.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}

In [ ]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [ ]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [ ]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

# 1.4 DataSet

In [ ]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [ ]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [ ]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

# 1.5 Search

## Indexing with minsearch


In [ ]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

## Trying a search

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

## Boosting fields

In [ ]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

## Filtering by course

In [ ]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [ ]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

## Wrapping it in a function

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

# 1.6 Building the Prompt


## Instructions

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

## The user prompt template


In [ ]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

## Building the context

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

## Building the prompt


In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [ ]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

# 1.7 The LLM

## Sending the prompt to the LLM


In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

## Exploring the response


In [ ]:
response.output[0]

ResponseOutputMessage(id='msg_00a430921ef334be006a3f819df0c081968df9e2500420c87e', content=[ResponseOutputText(annotations=[], text='Yes — you can join now and start learning.\n\nIf you want a certificate, make sure to submit your project while the submission form is still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [ ]:
response.output[0].content[0].text

'Yes — you can join now and start learning.\n\nIf you want a certificate, make sure to submit your project while the submission form is still open.'

In [ ]:
response.output_text

'Yes — you can join now and start learning.\n\nIf you want a certificate, make sure to submit your project while the submission form is still open.'

In [ ]:
response.usage

ResponseUsage(input_tokens=480, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=34, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=514)

## Calculating the price


In [ ]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.000513

## Message history


In [ ]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

## The LLM function


In [ ]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

## Full RAG


In [ ]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [ ]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. If you want a certificate, make sure to submit your project while submissions are still open.


In [ ]:
rag("How do I get a certificate?")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.4-mini in organization org-gSJCDct1Mm7TXK9ly8OLI3td on requests per day (RPD): Limit 50, Used 50, Requested 1. Please try again in 28m48s. Visit https://platform.openai.com/account/rate-limits to learn more. You can increase your rate limit by adding a payment method to your account at https://platform.openai.com/account/billing.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}

# 1.8 RAG Helper

## ingest.py

In [ ]:
import requests
from minsearch import Index

def load_faq_data():
    docs_url = "https://datatalks.club/faq/json/courses.json"
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = "https://datatalks.club/faq"

    for course in courses_raw:
        course_url = f"""{url_prefix}{course["path"]}"""
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()

        documents.extend(course_data)

    return documents

def build_index(documents):
    index = Index(
        text_fields=["question", "section", "answer"],
        keyword_fields=["course"]
    )
    index.fit(documents)
    return index

## rag_helper.py


In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

In [ ]:
class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course="llm-zoomcamp",
        model="gpt-5.4-mini"
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        boost_dict = {"question": 3.0, "section": 0.5}
        filter_dict = {"course": self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )
    
    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc["section"])
            lines.append("Q: " + doc["question"])
            lines.append("A: " + doc["answer"])
            lines.append("")

        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )
    
    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer

## Using it in a notebook


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now. If you want a certificate, make sure you submit your project while submissions are still open.


In [ ]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [ ]:
assistant.rag("How do I get a certificate?")

'You can get a certificate if you **finish the course with a live cohort** and **pass the Capstone project**.  \n\nCertificates are **not awarded for the self-paced mode**, because you need to **peer-review 3 capstones** during the course while it is running.'

In [ ]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join the course after it started. You can start whenever you want, and the materials are available.\n\nIf you want a certificate, you must finish the course with a live cohort and submit your project while submissions are still open.'

# 1.9 Data Ingestion

In [ ]:
# uv add sqlitesearch
%pip install sqlitesearch

Note: you may need to restart the kernel to use updated packages.


d:\Projects\LLM-Zoomcamp\.venv\Scripts\python.exe: No module named pip


## Ingestion notebook


In [ ]:
from ingest import load_faq_data

documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Loaded 1350 documents


In [ ]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 85 documents


In [ ]:
import time
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

for doc in docs_llm:
    index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    time.sleep(0.5)

index.close()
print("Done. Index saved to faq.db")

Added: I just discovered the course. Can I still join?...
Added: Course: I have registered for the LLM Zoomcamp. When can I e...
Added: What is the video/zoom link to the stream for the “Office Ho...
Added: How should I start the course and follow the weekly workflow...
Added: Leaderboard: I am not on the leaderboard / how do I know whi...
Added: Certificate: Can I follow the course in a self-paced mode an...
Added: I missed the first homework - can I still get a certificate?...
Added: Homework: Why does the content keep changing?...
Added: When will the course be offered next?...
Added: Are there any lectures/videos? Where are they?...
Added: Where can I track the LLM Zoomcamp syllabus, deadlines, home...
Added: Are there live sessions or office hours for each module?...
Added: Can I use Bluesky for learning in public credits?...
Added: Where is the LLM Zoomcamp Telegram channel?...
Added: Why are we not using Langchain in the course?...
Added: OpenAI: Error when running OpenAI respon

## Querying notebook


In [ ]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [ ]:
sqlite_index.count()

340

In [ ]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?']

## RAG with sqlitesearch


In [ ]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

openai_client = OpenAI()
load_dotenv()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

In [ ]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join after the course has started. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [ ]:
sqlite_index.close()

## Comparing the two approaches


# 1.10 RAG Next Steps


In Part 1 of this module, we:

- Learned what RAG is and why it matters: retrieve documents, build a
  prompt, let the LLM generate a grounded answer
- Built a search engine over a real FAQ dataset using minsearch
- Created a prompt template that combines the user's question with
  search results
- Wired search + prompt + LLM into a working RAG pipeline
- Split ingestion and query into separate processes with sqlitesearch

You now have a working RAG system and a clear mental model for how each
piece fits together. From here, the work is making each piece better.

## Two directions forward

[Part 2 of this module: Agents](11-agents-intro.md). Our pipeline runs
search once with the exact user query, unchanged. If that search
returns garbage, the LLM has no way to recover. An agent puts the LLM
in charge instead. It decides what to search for, how many searches to
run, and when to stop.

An agent also handles questions in another language. It translates the
query before searching, then translates the answer back afterward.

[Module 2: Vector Search](../../02-vector-search/). Keyword matches
are exact. Vector search matches by semantic meaning instead, which
helps when the user phrases things differently from the FAQ.

## Elasticsearch

Elasticsearch is the industry standard for text search.

It supports:

- Full-text search with BM25
- Filtering, aggregations, and faceting
- Vector search (dense and sparse)
- Distributed scaling
- Real-time indexing

It's heavier than sqlitesearch but handles production workloads at
scale. If you're building a real RAG system, Elasticsearch (or
OpenSearch) is a common choice for the search backend.

For an Elasticsearch tutorial, see the
[supplementary materials for Module 1](../../cohorts/2025/01-intro/elastic-search.md).

## Fine-tuning vs RAG

Instead of retrieving documents at query time, you could fine-tune
the LLM on your data.

Fine-tuning means taking a model's weights and adjusting them for
your specific use case.

This works, but it has downsides:

- It requires special hardware (GPUs) and tools we don't cover in
  this course
- It's difficult to update when new data arrives - you don't want to
  retrain the model every time a new FAQ entry is added
- The LLM already has internal knowledge, but RAG gives it access to
  information it wasn't trained on

RAG is more flexible, cheaper, and works with any LLM. In practice,
fine-tuning is rarely needed. I've never personally hit a case that
required it. When I analyzed around 2,500 job descriptions for my AI
engineering field guide, few asked for it. So focus on RAG first, and
reach for fine-tuning only when you genuinely need it.

## Moving forward

Try these next steps:

- Try different prompts and see how the answers change
- Add more data sources to the knowledge base
- Experiment with different LLM models (GPT-4o, Claude, Gemini, local
  models via Ollama)
- Try Elasticsearch as a search backend


# 1.11 Agents

In Part 1 of this module we built RAG pipelines.

Every pipeline we wrote followed the same flow:

- search the FAQ,
- build a prompt with the results,
- send it to the LLM.

This returns good answers when the user's query matches the documents.
The search finds the right entry, the LLM reads it, and you get a
helpful reply.

Often, though, the search returns nothing useful.

- Maybe the user made a typo.
- Maybe they asked the question in an unusual way.
- Maybe they need information from two different searches.

We use lexical search here, so the search looks for an exact match.
One typo and it misses the entry it needed. In our pipeline there's
no recovery. The search runs once, and if it returns garbage the LLM
gets garbage. Our pipeline always does the same thing, no matter what.

Instead of routing the user question straight to search, we can hand
control to the LLM and let it drive.

The LLM is in charge now, and it can:

- fix typos
- search again with different terms
- ask the user a clarifying question

A fixed flow can't do any of this. Once we put the LLM in control,
our system becomes agentic, so it's flexible rather than rigid.

An agent uses an LLM to decide which actions to take and in which
order. Instead of a fixed flow, the LLM chooses what to do at each
step.

In Part 2 of this module, we'll cover:

- Function calling, so we can give the LLM tools it can use
- The agentic loop, where the LLM decides when to call a tool, when
  to call another one, and when to stop and answer
- Frameworks, the libraries that run this loop for you

We build on top of the RAG pipeline from Part 1, which used keyword
search with minsearch. If you skipped Part 1, the next lesson does a
quick revision and walks you through downloading the helpers.


# 1.12 Quick RAG Revision (Optional)


Before we talk about agents, let's set up the RAG pipeline we built
in Part 1.

Our courses have a lot of participants. They ask the same questions
over and over, so we keep a FAQ document and point students to it. RAG
takes that FAQ and finds the entry that matches a question. It then
sends the entry to an LLM so it can answer. That way a student gets a
reply right away instead of scrolling through a long document.

We'll use two helpers we defined earlier in this module:

- [`rag_helper.py`](../code/rag_helper.py) - the `RAGBase` class wrapping search, prompt building, and the LLM call
- [`ingest.py`](../code/ingest.py) - `load_faq_data` and `build_index` for loading the FAQ and building a minsearch index

If you're working through Part 2 as a standalone workshop (without
Part 1), download them into your project:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
```

## Setting up RAG

Set up the OpenAI client:

```python
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()
```

Load the data and build the search index:

```python
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)
```

Create the assistant:

```python
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)
```

## Testing it

Let's try a question:

```python
assistant.rag("How do I run Ollama locally?")
```

This works fine. The search finds relevant FAQ entries about Ollama,
and the LLM gives a good answer.

Now try something slightly different:

```python
assistant.rag("How do I run Olama locally?")
```

The word "Olama" doesn't match "Ollama" in our index. We use lexical
search, so it looks for the exact word and finds nothing. The LLM
gets these bad results and either says "I don't know" or answers with
irrelevant information.

This is the limitation of a fixed pipeline. The search runs once with
the exact query the user typed, and there's no second chance. The
pipeline doesn't know the search failed, so it can't try again with a
corrected query.

We need something smarter. We need an agent.


# 1.13 Function Calling


In the previous lesson we built a RAG pipeline with `RAGBase.rag()`
and saw it fail on the "Olama" typo. The search returned nothing
useful, and the LLM had no way to recover.

The flow that broke:

```mermaid
flowchart TD
    U([User: How do I run Olama?])
    S[search - Olama - no useful results]
    A([LLM: I don't have information about Olama.])

    U --> S --> A
```

The pipeline is fixed: search, build prompt, LLM.

```python
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer
```

The LLM is a passenger here, not a driver. It never sees the bad
search results, so it can't try again with a corrected query.

## The agent alternative

An agent puts the LLM in charge.

Instead of running search ourselves, we give the LLM a `search` tool.
It decides when to call it and what to search for.

The same typo question now goes like this:

```mermaid
flowchart TD
    U([User: How do I run Olama?])
    L1[LLM: I'll search for 'Olama']
    S1[search - Olama - no useful results]
    L2[LLM: Hmm, no results. Maybe a typo for 'Ollama'?]
    S2[search - Ollama - found results!]
    A([LLM: Here's how to run Ollama locally...])

    U --> L1 --> S1 --> L2 --> S2 --> A
```

The LLM searched, saw the results were bad, and decided to try again
with a different query. It made that decision on its own. We didn't
write any code to handle typos.

The difference is about who makes the decisions:

- With RAG, the developer decides. We fix the steps up front, so
  search always runs once with the exact user query.
- With an agent, the LLM decides. It chooses which actions to take
  and when to stop.

The mechanism that makes this possible is function calling, and that's
what the rest of this lesson is about.

## Asking without tools

First, let's see what the LLM does without any tools. We ask it a
course-specific question and look at the answer.


In [ ]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—probably, but it depends on the course’s enrollment policy and whether registration is still open.\n\nIf you want, I can help you figure out the fastest way to ask or check. A simple message could be:\n\n> Hi, I just discovered this course and I’m interested in joining. Is it still possible to enroll?\n\nIf you’d like, I can also help you write a more formal or more casual version.'

## Defining the tool

First we define a top-level `search` function that queries the `index`
directly. The model will reference it by this name. We keep the Python
function and the tool name aligned so the dispatch is easier later.


In [ ]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.



In [ ]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

The `description` is the most important field, because the model reads
it to decide when to call the function. `parameters` is a JSON schema
for the arguments, and we mark `query` as required so the model always fills it in.

## Sending the question with the tool

Now we send the same question as before, but this time we include the
tool in the request:


In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join late enrollment eligibility"}', call_id='call_tUOi7voJtfZFGn5DAz0R6H7S', name='search', type='function_call', id='fc_043cc9954aae1a81006a3f7ee9c4ec8193aa482eb6dc68c76b', namespace=None, status='completed')]

Look at the output. Instead of a message with the answer, the response
contains a `function_call` entry. The model decided it needs to search
the FAQ before answering. Rather than reply, it asked us to run the
search function first.

Look at the arguments too. The model didn't pass our question
verbatim. It judged the raw question wasn't the best query to search
with. So it rewrote our enrollment question into search keywords like
"enroll late join course".

## Executing the function and sending the result back

The function call contains JSON arguments. We parse them, call our
`search` function, and serialize the result.


In [ ]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

Now we send this result back to the model. First, we add the model's
output to the conversation history - the model needs to see its own
function call. Then we add the tool result.


In [ ]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

The `call_id` links the tool result to the specific function call the
model requested. If the model makes multiple function calls in one
turn, each one gets its own `call_id`.

## Asking the model again

We call the API a second time with the expanded history:


In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes, you can join. If you want to receive a certificate, you need to submit your project while submissions are still open.'

This time the model has the original question, its own decision to
call `search`, and the FAQ results. It can now produce a proper
course-specific answer.

We have to send the whole history because LLMs are stateless between
API calls. The memory is the list you send as `input`. If you send
only the tool result, the model has no idea what's going on. So on
this second call we replay everything we have so far. That means the
question, the decision to call `search`, and the result we got back.

That's the full function-calling loop for a single turn. With plain
RAG we made one call, and here we make two. Turning RAG agentic means more round-trips.

People call this pattern "agentic RAG", "tool use", or "function
calling". The idea behind all of them is the same. The LLM decides
which tools to call.

## Token usage and cost

We just made two API calls instead of one. Each call we send to the
model costs money, so it's worth checking how much one tool-using turn
actually costs.

The response has a `usage` field with the token counts:


In [ ]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


This usage is only for the second API call. The first call also has
its own usage and its own cost. That was the call where the model
decided to invoke `search`. Two calls means we pay twice. We pay even more on the second call, because we resend the full history as input.

With a real agent loop the model can make many calls, so the costs add up. Keep an eye on `usage` while you develop.


# 1.14 The Agentic Loop


In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back, and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the instructions, the better the
  agent helps.
- Tools, the functions the agent can call to carry out the task. For
  us that's only `search`.
- Memory, the message history. We append every prompt, every model
  output, and every tool result. The agent reads this to know what it
  has already tried.

Everything below is the code that wires these three together inside a loop.

## A developer prompt

So far we've relied on the model to figure out when to search. We make that more reliable with a `developer` message that spells out how to
behave. This is where we give the agent its role. The same message
also pushes it toward multiple searches, so we get to watch the loop
run more than once.

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

## A function-call helper

We'll be running function calls repeatedly inside the loop, so let's
wrap that in a small helper. It turns the JSON arguments into a Python
dict, calls the right function, and serializes the result. We only
have one tool for now, so we dispatch on the function name directly.


In [ ]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

The helper returns the exact structure the Responses API expects.
When we add more tools later, we'll extend this with more `if`
branches (or switch to a registry).

## Processing one response

Let's process a single model response. We append each output entry to
the conversation, print any messages, and run any function calls.
Function-call results get appended too.

In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join the course enrollment late discover course can I join"}
function_call: search {"query":"course registration enrollment open can I still join"}
function_call: search {"query":"new student join course after start FAQ"}


The `has_function_calls` flag tells us whether the model needs another API call. If the response contains a function call, the updated `messages` has tool output the model hasn't seen yet. We'll need to send it back.

## The full agent loop

We wrap this in a `while` loop. The loop keeps calling the model until it returns a response without any function calls. We also keep an iteration counter so we can see how many round-trips happened.


In [ ]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. You can also start learning and doing homework right away; registration is mainly for interest tracking, and you don’t need a confirmation email to begin.

If you’d like, I can also help with how to start the course or how the weekly workflow works.


This is the core agent loop. The model reasons about the next action. Your code performs it, and the model sees the result on the next turn. The loop stops when the model returns a final answer with no more tool
calls.

We don't decide how many times the model searches. The model does,
and we keep looping until it stops asking for tools.

The exit condition is the simplest one possible. No function calls
this turn means we're done. Other frameworks add safety nets on top,
like a max iteration count, a token budget, or a wall-clock limit. You might cap it at five iterations and force an answer on the last one. The core is still this one flag.

## Wrapping it in a function

Let's wrap the loop in a function so we can reuse it. The function
takes the instructions and the question as parameters, and returns
the final answer.

In [ ]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

Try it with a question that has a typo:


In [ ]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local model FAQ Ollama"}
function_call: search {"query":"run Ollama locally installation start service FAQ"}
function_call: search {"query":"Ollama local setup use models FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. Install Ollama from: https://ollama.com/download  
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Start a local model in your terminal:
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat interface.

3. Test that the local server is running:
   ```bash
   curl http://localhost:11434
   ```

4. If you want to use it from Python:
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )


'To run Ollama locally:\n\n1. Install Ollama from: https://ollama.com/download  \n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Start a local model in your terminal:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a local chat interface.\n\n3. Test that the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. If you want to use it from Python:\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you meant a different “Olama” setup or want help with Docker/Python integration, let me know—are there other areas you want to explore?'

Watch what happens. The agent searches for "Olama" and gets poor
results. It then searches again with "Ollama" and finds the answer.
The loop lets the model recover from a bad search on its own. That's
the whole point of going agentic.

Also try the course enrollment question:


In [ ]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"Can I still join the course if I just discovered it enrollment late join FAQ"}
function_call: search {"query":"late enrollment join course after start FAQ"}
function_call: search {"query":"join the course after it starts course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course even if you just discovered it.

A couple of notes:
- You can start learning anytime; the videos and GitHub materials are available.
- If you want a certificate, you’ll need to submit your project while submissions are still open.

If you want, I can also help you figure out the best way to start catching up. Are there other areas you want to explore?


'Yes — you can still join the course even if you just discovered it.\n\nA couple of notes:\n- You can start learning anytime; the videos and GitHub materials are available.\n- If you want a certificate, you’ll need to submit your project while submissions are still open.\n\nIf you want, I can also help you figure out the best way to start catching up. Are there other areas you want to explore?'

## Encouraging multiple searches

There's a subtle issue here. The model often answers after the first
search, even when more searches would help. It reasons that it already knows enough, so why bother. We push it to explore more by rewriting the instructions.


In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course late discovered can I join enrollment late registration FAQ"}
iteration #2...
function_call: search {"query":"certificate project submission still accepting submissions course FAQ project submit while accepting submissions"}
iteration #3...
ASSISTANT:
Yes — you can still join the course even if you just discovered it.

One important note: if you want to receive a certificate, you need to submit your project while submissions are still open.

If you’d like, I can also explain how registration, homework submission, and certification work. Is there anything else you want to explore?


'Yes — you can still join the course even if you just discovered it.\n\nOne important note: if you want to receive a certificate, you need to submit your project while submissions are still open.\n\nIf you’d like, I can also explain how registration, homework submission, and certification work. Is there anything else you want to explore?'

Now the agent makes multiple searches per question and doesn't stop
after the first round of results. The instructions are how we steer
the agent. It can still decide to skip ahead sometimes, so don't
expect it to follow them every single run.

## Restricting off-topic questions

Right now the agent will answer anything. Ask it about chess and it
will still try.


In [ ]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit definition chess opening queen's gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening what is queen gambit"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a classic **chess opening**.

It starts with:
1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a gambit to try to get control of the center and encourage Black to weaken their position.

A quick idea of the opening:
- **White** aims for central control and active piece development
- **Black** can accept the pawn or decline it
- It’s one of the most famous and heavily studied openings in chess

If you want, I can also explain:
- the **main idea behind the Queen’s Gambit**
- the difference between **Queen’s Gambit Accepted** and **Declined**
- or show a **simple move-by-move example**


'The **Queen’s Gambit** is a classic **chess opening**.\n\nIt starts with:\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** as a gambit to try to get control of the center and encourage Black to weaken their position.\n\nA quick idea of the opening:\n- **White** aims for central control and active piece development\n- **Black** can accept the pawn or decline it\n- It’s one of the most famous and heavily studied openings in chess\n\nIf you want, I can also explain:\n- the **main idea behind the Queen’s Gambit**\n- the difference between **Queen’s Gambit Accepted** and **Declined**\n- or show a **simple move-by-move example**'

We want a course assistant, not a general chatbot. We tighten the
instructions so the agent only answers from the FAQ. For our own use
we might be fine letting it answer from general knowledge. So treat
this mainly as an illustration of steering scope.


In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"chess opening queen's gambit course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find any FAQ entry about “queen gambit,” so it looks like this isn’t a course/logistics question.

If you meant the course, feel free to ask a course-related question and I’ll look it up. Are there other areas you want to explore?


'I couldn’t find any FAQ entry about “queen gambit,” so it looks like this isn’t a course/logistics question.\n\nIf you meant the course, feel free to ask a course-related question and I’ll look it up. Are there other areas you want to explore?'

This is a lightweight form of an input guardrail. We tell the agent
what's in scope and what isn't. A real guardrail checks the input
before the agent runs and can block off-topic questions outright.
That's a separate topic, but instructions are the first place to start.

This handwritten loop is the best way to understand what frameworks
hide from you. Every agent framework wraps this same pattern, whether it's LangChain, PydanticAI, or the OpenAI Agents SDK.


# 1.15 Toy AI Kit

# ToyAIKit

The handwritten agent loop from the previous lesson is educational but repetitive. Every time you build a new agent, you'd write the same while-loop, the same function-call handling, the same message
management.

ToyAIKit wraps this pattern so you can focus on tools, prompts, and
behavior. We built it together in a DataTalks.Club workshop a while
back. It does the same thing as our handwritten loop with less
boilerplate. If you open its `runners` code, you'll find the same
`while True` loop we wrote by hand.

I use it here on purpose, because I don't want to pick a winner among the production frameworks. ToyAIKit is small and easy to read, so when something breaks you can see exactly what happened. That makes it handy for developing and debugging locally before you go to production.

One caveat. ToyAIKit is a teaching and experimentation library, and it is NOT meant for production use. We use it because it's minimal and you can see what it does.



## Setup

Install it:

```bash
uv add toyaikit
```

Import the classes we need:


In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback




```python
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
```

## Registering the tool

We register our `search` function along with the schema from earlier
lessons:

```python
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
```

## Letting ToyAIKit generate the schema

Writing that schema by hand is annoying, and we don't want to do it
for every function. So we don't have to.

If we add a type hint and a docstring to `search`, ToyAIKit reads them
and derives the schema for us:

```python
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )
```

Then register it without passing a schema:

```python
agent_tools = Tools()
agent_tools.add_tool(search)
```

You can look at what ToyAIKit produced.

```python
agent_tools.get_tools()
```

The output is the same JSON schema we hand-wrote in the function
calling lesson. ToyAIKit generated it from the docstring and the type
hint.

Every modern agent framework does this same trick. It reads a typed
Python function with a docstring and builds the schema from it. The
OpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this
way. You write the tool and the framework figures out how to describe
it.

## The chat interface and runner

Create the chat interface and a callback, then build the runner:

```python
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)
```

The `chat_interface` handles display in the notebook. The `callback`
renders model messages and tool calls as they happen. The runner runs
the agent loop, the same `while True` we wrote by hand. It sends
messages, executes function calls, adds tool outputs back, and repeats
until the model is done.

We pick `gpt-5.4-mini` here on purpose. Without it, ToyAIKit falls
back to a smaller, faster default that doesn't follow the instructions
as reliably.

## Running one prompt

Run a single prompt:

```python
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)
```

We used the typo "Olama" on purpose. The agent searches and gets poor
results, then retries with "Ollama". The recovery is the same as the
handwritten loop. The notebook output is nicer to watch. Each tool
call and message renders inline, so you can look at every search
result.

The `result` is a `LoopResult` with `all_messages` (the full
conversation), token counts, and `cost` (computed from token usage).

## Cost and tokens

Look at what the call cost:

```python
result.cost
```

Useful while developing - especially with multi-turn agents where one
prompt can trigger several model calls. The handwritten loop made you
compute this by hand. The framework keeps a running total for you.

You can also look at the full message history.

```python
result.all_messages
```

This is just a list - the same `messages` list we maintained by hand.

## Continuing the conversation

Take the messages from the previous result and pass them as
`previous_messages` on the next `loop` call:

```python
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)
```

The runner picks up where the last call left off, with the same agent
loop and an extended history. The model knows "different model" refers
to Ollama because it sees the previous turn in memory. Without that
history, it would have no idea what we're asking about.

## Interactive chat

For a chat-like workflow, run the built-in input loop:

```python
runner.run()
```

Type questions and get answers. Type "stop" to exit.
